## Agent: The Daily Dish Chatbot

"The Daily Dish," a popular restaurant with a customer service problem. Chef Maria, the owner, needs your help: "My team spends hours answering the same questions about reservations and menus," she explains. "Build a chatbot that feels personal, to handle these inquiries so my staff can focus on creating exceptional dining experiences." Chef Maria sends you a PDF of frequently asked questions (FAQs) and now you must get the job done!

### **Project overview: Our chatbot's workflow**

Our chatbot will follow a four-step process to answer a user's question:

1. **Query Understanding Phase (Agent 1):**
The user’s question is first handled by a query understanding agent. This agent’s role is to interpret the user’s intent and extract key details from the question.

2. **Document Retrieval Phase (Agent 2):**
The interpreted query is used to search the restaurant’s FAQ PDF. This agent’s role is to retrieve only the most relevant sections of the document.

3. **Memory Phase (Agent 3):**
The retrieved information and user interaction are passed to a memory agent. This agent’s role is to store and recall conversation context, supporting more personalized responses.

4. **Response Generation Phase (Agent 4 – LLM):**
The retrieved FAQ content, along with memory context, is sent to an LLM-based response agent. This agent’s role is to generate a clear, friendly, and complete answer for the customer.

```
┌──────────────┐
│   User       │
│ (Question)   │
└──────┬───────┘
       │
       ▼
┌───────────────────────────┐
│ Query Understanding       │
│ Agent (Agent 1)           │
│ - Interprets user intent  │
│ - Extracts key keywords   │
└──────┬────────────────────┘
       │
       ▼
┌───────────────────────────┐
│ Document Retrieval        │
│ Agent (Agent 2)           │
│ - Searches FAQ PDF        │
│ - Retrieves relevant text │
└──────┬────────────────────┘
       │
       ▼
┌───────────────────────────┐
│ Memory Agent              │
│ (Agent 3)                 │
│ - Stores past questions   │
│ - Recalls conversation    │
│   context                 │
└──────┬────────────────────┘
       │
       ▼
┌───────────────────────────┐
│ LLM Response Agent        │
│ (Agent 4)                 │
│ - Combines retrieved info │
│ - Uses memory context     │
│ - Generates friendly reply│
└──────┬────────────────────┘
       │
       ▼
┌──────────────┐
│   User       │
│ (Final Reply)│
└──────────────┘
```


---


#### **Step 1 — Install Prerequisites**


<span style="color:red"><b>Restart the kernel once the installation is complete.</b></span>


### **Conclusion**

Congratulations! You've successfully built a basic RAG chatbot from scratch using Python and the OpenAI API. You've seen how to implement a **retrieval agent** to find relevant context and a **generative agent** to craft a final response. This hands-on experience provides a strong foundation for understanding more complex AI agent frameworks and building your own custom LLM-powered applications.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pypdf PyPDF2 scikit-learn nltk openai

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# ============================================================
# THE DAILY DISH — FINAL RAG CHATBOT
# ============================================================

# Install Required Libraries First:
# !pip install PyPDF2 scikit-learn nltk openai

# ============================================================
# STEP 1 — IMPORT LIBRARIES
# ============================================================

import re
import nltk
import numpy as np

from PyPDF2 import PdfReader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK tokenizer
nltk.download('punkt')
nltk.download('punkt_tab')

# ============================================================
# STEP 2 — LOAD PDF
# ============================================================

# Google Drive PDF Path
faq_pdf_path = "/content/drive/MyDrive/The_Daily_Dish_FAQ.pdf"

reader = PdfReader(faq_pdf_path)

full_text = ""

for page in reader.pages:

    text = page.extract_text()

    if text:
        full_text += text + "\n"

print("PDF Loaded Successfully!")
print("-" * 50)

# ============================================================
# STEP 3 — CLEAN TEXT
# ============================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'[^a-zA-Z0-9?.!, ]', '', text)

    return text

cleaned_text = clean_text(full_text)

# ============================================================
# STEP 4 — CREATE FAQ QUESTION-ANSWER PAIRS
# ============================================================

faq_pairs = []

# Split based on numbered questions
sections = re.split(r'\d+\.\s*q', cleaned_text)

for section in sections:

    # Ensure section contains answer part
    if "a" in section:

        # Better split for Q and A
        parts = re.split(r'\ba\b', section, maxsplit=1)

        if len(parts) == 2:

            question = parts[0].strip()

            answer = parts[1].strip()

            faq_pairs.append({
                "question": question,
                "answer": answer
            })

print(f"Total FAQ Pairs Created: {len(faq_pairs)}")
print("-" * 50)

# ============================================================
# STEP 5 — CREATE TF-IDF VECTORS
# ============================================================

vectorizer = TfidfVectorizer()

questions = [faq["question"] for faq in faq_pairs]

question_vectors = vectorizer.fit_transform(questions)

print("TF-IDF Vectorization Complete!")
print("-" * 50)

# ============================================================
# AGENT 1 — QUERY UNDERSTANDING AGENT
# ============================================================

def query_understanding_agent(user_query):

    user_query = clean_text(user_query)

    keywords = user_query.split()

    return {
        "original_query": user_query,
        "keywords": keywords
    }

# ============================================================
# AGENT 2 — DOCUMENT RETRIEVAL AGENT
# ============================================================

def retrieval_agent(processed_query):

    query_vector = vectorizer.transform(
        [processed_query["original_query"]]
    )

    similarities = cosine_similarity(
        query_vector,
        question_vectors
    )

    best_match_index = np.argmax(similarities)

    best_score = similarities[0][best_match_index]

    retrieved_faq = faq_pairs[best_match_index]

    return {
        "question": retrieved_faq["question"],
        "answer": retrieved_faq["answer"],
        "similarity_score": best_score
    }

# ============================================================
# AGENT 3 — MEMORY AGENT
# ============================================================

conversation_memory = []

def memory_agent(user_query, bot_response):

    conversation_memory.append({
        "user": user_query,
        "bot": bot_response
    })

    # Keep last 5 conversations only
    if len(conversation_memory) > 5:
        conversation_memory.pop(0)

def get_memory_context():

    context = ""

    for chat in conversation_memory:

        context += f"User: {chat['user']}\n"
        context += f"Bot: {chat['bot']}\n"

    return context

# ============================================================
# AGENT 4 — RESPONSE GENERATION AGENT
# ============================================================

def response_generation_agent(user_query, retrieved_data):

    similarity_score = retrieved_data["similarity_score"]

    # Low similarity protection
    if similarity_score < 0.10:

        return (
            "I'm sorry, I couldn't find relevant information "
            "related to your question."
        )

    # Extract exact answer
    answer = retrieved_data["answer"]

    # Clean formatting
    answer = re.sub(r'\s+', ' ', answer)

    answer = answer.strip()

    # Capitalize nicely
    answer = answer.capitalize()

    return answer

# ============================================================
# MAIN CHATBOT LOOP
# ============================================================

print("\n")
print("==========================================")
print("WELCOME TO THE DAILY DISH CHATBOT")
print("Type 'exit' to stop chatting")
print("==========================================")

while True:

    user_query = input("\nYou: ")

    # Exit condition
    if user_query.lower() == "exit":

        print("\nBot: Thank you for visiting The Daily Dish!")
        break

    # ----------------------------------------
    # AGENT 1 — QUERY UNDERSTANDING
    # ----------------------------------------

    processed_query = query_understanding_agent(user_query)

    # ----------------------------------------
    # AGENT 2 — DOCUMENT RETRIEVAL
    # ----------------------------------------

    retrieved_data = retrieval_agent(processed_query)

    # ----------------------------------------
    # AGENT 4 — RESPONSE GENERATION
    # ----------------------------------------

    bot_response = response_generation_agent(
        user_query,
        retrieved_data
    )

    # ----------------------------------------
    # AGENT 3 — MEMORY STORAGE
    # ----------------------------------------

    memory_agent(user_query, bot_response)

    # ----------------------------------------
    # FINAL RESPONSE
    # ----------------------------------------

    print("\nBot:", bot_response)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


PDF Loaded Successfully!
--------------------------------------------------
Total FAQ Pairs Created: 22
--------------------------------------------------
TF-IDF Vectorization Complete!
--------------------------------------------------


WELCOME TO THE DAILY DISH CHATBOT
Type 'exit' to stop chatting

You: What are your opening hours?

Bot: The daily dish is open monday to friday from 1100 am to 1000 pm, and on saturday and sunday from 1000 am to 1100 pm.

You: Can I reserve a table for 8 people?

Bot: Celebration? a we do offer a selection of delicious desserts. however, if you wish to bring your own cake, a small plating fee of 15 will apply.

You: reservation of people ?

Bot: The daily dish is open monday to friday from 1100 am to 1000 pm, and on saturday and sunday from 1000 am to 1100 pm.

You: Do you offer vegan food?

Bot: Absolutely! we have a dedicated section on our menu for vegetarian and vegan dishes, and many other items can be modified. just ask your server.

You: Where 

KeyboardInterrupt: Interrupted by user